# 02 — Feature Engineering

This notebook creates leakage-aware, model-friendly features from the raw interaction table. The goal is to keep the transformation simple enough to explain in an interview.

In [2]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/train_de.csv")
OUTPUT_PATH = Path("../data/processed/ctr_features.csv")
SAMPLE_ROWS = 250_000

df = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS)
df = pd.read_csv(DATA_PATH, sep="\t", nrows=SAMPLE_ROWS)
df["utcdate"] = pd.to_datetime(df["utcdate"], errors="coerce")


In [4]:
# Time/context features
df["hour"] = df["utcdate"].dt.hour.fillna(-1).astype(int)
df["dayofweek"] = df["utcdate"].dt.dayofweek.fillna(-1).astype(int)
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

# Keep the raw identifiers as categorical variables.
# LightGBM receives them through an OrdinalEncoder in notebook 03.
feature_cols = [
    "userid", "offerid", "countrycode", "category", "merchant",
    "hour", "dayofweek", "is_weekend", "click"
]
feature_cols = [col for col in feature_cols if col in df.columns]
features = df[feature_cols].copy()
for col in ["userid", "offerid", "countrycode", "category", "merchant"]:
    features[col] = features[col].astype(str)

features.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(features):,} rows to {OUTPUT_PATH}")
features.head()


Saved 250,000 rows to ..\data\processed\ctr_features.csv


,userid,offerid,countrycode,category,merchant,hour,dayofweek,is_weekend
0,fa937b779184527f12e2d71c711e6411236d1ab59f8597...,c5f63750c2b5b0166e55511ee878b7a3,de,100020213,f3c93baa0cf4430849611cedb3a40ec4094d1d370be841...,17,1,0
1,f6c8958b9bc2d6033ff4c1cc0a03e9ab96df4bcc528913...,19754ec121b3a99fff3967646942de67,de,100020213,21a509189fb0875c3732590121ff3fc86da770b0628c18...,17,1,0
2,02fe7ccf1de19a387afc8a11d08852ffd2b4dabaed4e2d...,5ac4398e4d8ad4167a57b43e9c724b18,de,125801,b042951fdb45ddef8ba6075ced0e5885bc2fa4c4470bf7...,17,1,0
3,9de5c06d0a16256b13b8e7cdc50bf203ecef533eb5cbe1...,be83df9772ec47fd210b28091138ff11,de,125801,4740b6c83b6e12e423297493f234323ffd1c991f3d4496...,17,1,0
4,8d26ade603ea5473c3844aebfcd9e96e6adc8ff411576e...,3735290a415dc236bacd7ed3aa03b2d5,de,125801,8bf8f87492a799528235c04bb18ff2d12db5058ff6e9a0...,17,1,0


## Leakage note

A common CTR mistake is calculating user/ad CTR from the entire dataset and then feeding that statistic into the same rows used to train the model. That lets future information leak backward in time.

For a production system, historical CTR features should be generated with a rolling/expanding window and shifted so that each impression only sees information available before it happened. This project keeps the first version intentionally simple and calls out that production improvement explicitly.
